In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Regresion Lineal Multiple

La **regresion lineal simple** (capitulo anterior) modela la respuesta con un solo
predictor. En la practica, los procesos dependen de varios factores simultaneamente.
La **regresion lineal multiple** (MLR) extiende el modelo a $p$ predictores:

$$Y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \varepsilon$$

En este capitulo construiremos el modelo paso a paso, primero a mano y luego con
`statsmodels`, usando el mismo caso de estudio del capitulo anterior enriquecido
con un segundo predictor.

## Contenidos

1. **Caso de estudio** -- extendemos el dataset de catalizador con temperatura.
2. **El modelo MLR** -- ecuacion, supuestos y forma matricial.
3. **Estimacion OLS** -- derivacion manual (numpy) y verificacion (statsmodels).
4. **Bondad de ajuste** -- $R^2$ vs $R^2_{adj}$.
5. **Inferencia** -- F-test global e intervalos para coeficientes.
6. **Multicolinealidad** -- Factor de Inflacion de la Varianza (VIF).
7. **Seleccion de variables** -- eliminacion hacia atras.
8. **Diagnosticos** -- residuos, Q-Q, distancia de Cook.
9. **Prediccion** -- IC y IP con `get_prediction`.
10. **Sintesis y problemas de practica**.

---
## Caso de estudio: Concentracion y Temperatura vs. Rendimiento

En el capitulo anterior el ingeniero de procesos quimicos demostro que la
concentracion de catalizador $x_1$ (g/L) explica el 95.6% de la variabilidad
del rendimiento $y$ (%).
Sin embargo, hay una segunda variable controlable: la **temperatura de reaccion**
$x_2$ (grados C).

Se realizan 20 nuevos experimentos variando ambos factores **de forma independiente**
(temperatura aleatorizada entre 60 y 80 grados C, concentracion de 1 a 20 g/L).

> **Pregunta central:** agrega la temperatura informacion significativa
> sobre el rendimiento, mas alla de lo que ya explica la concentracion?

In [ ]:
np.random.seed(2026)
concentracion = np.arange(1, 21, dtype=float)
temperatura   = np.random.uniform(60, 80, 20)
epsilon       = np.random.normal(0, 4, 20)
rendimiento   = 20 + 2.5 * concentracion + 0.4 * temperatura + epsilon

df = pd.DataFrame({
    "concentracion": concentracion,
    "temperatura":   np.round(temperatura, 2),
    "rendimiento":   np.round(rendimiento, 2)
})

print("Dataset: Concentracion, Temperatura y Rendimiento")
print(df.round(2).to_string(index=False))
print()
print(f"n = {len(df)}")
print(f"Rendimiento: media = {rendimiento.mean():.2f}%,  DE = {rendimiento.std(ddof=1):.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150)

r_conc, _ = stats.pearsonr(concentracion, rendimiento)
axes[0].scatter(concentracion, rendimiento, color="steelblue", s=50)
axes[0].set_xlabel("Concentracion de catalizador (g/L)")
axes[0].set_ylabel("Rendimiento (%)")
axes[0].set_title(f"x1 vs y   (Pearson R = {r_conc:.3f})")
axes[0].grid(alpha=0.3)

r_temp, _ = stats.pearsonr(temperatura, rendimiento)
axes[1].scatter(temperatura, rendimiento, color="darkorange", s=50)
axes[1].set_xlabel("Temperatura de reaccion (grados C)")
axes[1].set_ylabel("Rendimiento (%)")
axes[1].set_title(f"x2 vs y   (Pearson R = {r_temp:.3f})")
axes[1].grid(alpha=0.3)

plt.suptitle("Asociacion individual de cada predictor con el rendimiento", fontsize=11)
plt.tight_layout()
plt.show()

r_x1x2, p_x1x2 = stats.pearsonr(concentracion, temperatura)
print(f"Correlacion entre x1 y x2: R = {r_x1x2:.4f}  (p = {p_x1x2:.3f})")
print("Los predictores son practicamente independientes: baja multicolinealidad esperada.")

---
## 1. El Modelo de Regresion Lineal Multiple

Con dos predictores el modelo es:

$$Y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \varepsilon, \qquad
\varepsilon \sim \mathcal{N}(0,\,\sigma^2)$$

- $\beta_0$: intercepto (rendimiento esperado cuando ambos predictores son cero).
- $\beta_1$: efecto *parcial* de $x_1$ (manteniendo $x_2$ fijo).
- $\beta_2$: efecto *parcial* de $x_2$ (manteniendo $x_1$ fijo).

**Los supuestos son los mismos que en SLR:** linealidad, independencia,
homocedasticidad y normalidad de los errores.

> **Interpretacion de los coeficientes:** en MLR cada $\hat{\beta}_j$ mide
> el cambio esperado en $y$ por una unidad de aumento en $x_j$,
> *ceteris paribus* (los demas predictores constantes).
> Este no es el mismo valor que obtendrıamos en una regresion simple de $y$ sobre $x_j$.

### 1.1 Forma matricial y estimador OLS

Con $n$ observaciones y $p$ predictores, el sistema se escribe como:

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}$$

donde $\mathbf{X}$ es la **matriz de diseno** $n \times (p+1)$ con una columna de unos
para el intercepto:

$$\mathbf{X} = \begin{pmatrix} 1 & x_{11} & x_{12} \\ 1 & x_{21} & x_{22} \\ \vdots & \vdots & \vdots \\ 1 & x_{n1} & x_{n2} \end{pmatrix}$$

El estimador de minimos cuadrados minimiza $\|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$
y tiene solucion cerrada (M\&R, ec. 12.9):

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$$

La covarianza de los estimadores es $\text{Cov}(\hat{\boldsymbol{\beta}}) = \sigma^2 (\mathbf{X}^\top\mathbf{X})^{-1}$,
de donde se extraen los errores estandar de cada coeficiente.

In [ ]:
n = len(rendimiento)

# Construir la matriz de diseno X: columna de unos, concentracion, temperatura
X_mat = np.column_stack([np.ones(n), concentracion, temperatura])

# Estimador OLS: beta_hat = (X'X)^{-1} X'y
XtX      = X_mat.T @ X_mat
Xty      = X_mat.T @ rendimiento
beta_hat = np.linalg.solve(XtX, Xty)   # equivalente a inv(X'X) @ X'y, pero mas estable

print("Matriz de diseno X (primeras 5 filas):")
print(pd.DataFrame(X_mat[:5], columns=["const", "conc", "temp"]).round(3).to_string(index=False))
print()
print("Estimador OLS:  beta_hat = (X'X)^{-1} X'y")
print(f"  beta_0_hat  = {beta_hat[0]:.4f}  (intercepto)")
print(f"  beta_1_hat  = {beta_hat[1]:.4f}  (concentracion)")
print(f"  beta_2_hat  = {beta_hat[2]:.4f}  (temperatura)")
print()
print(f"Modelo ajustado:  y_hat = {beta_hat[0]:.2f} + {beta_hat[1]:.2f}*x1 + {beta_hat[2]:.2f}*x2")
print()
print("Valores reales del modelo generador:")
print("  beta_0 = 20.00,  beta_1 = 2.50,  beta_2 = 0.40")

In [ ]:
fig = plt.figure(figsize=(9, 6), dpi=150)
ax  = fig.add_subplot(111, projection="3d")

ax.scatter(concentracion, temperatura, rendimiento,
           color="steelblue", s=50, zorder=5, label="Datos observados")

# Plano ajustado
conc_g = np.linspace(concentracion.min(), concentracion.max(), 20)
temp_g = np.linspace(temperatura.min(), temperatura.max(), 20)
cc, tt = np.meshgrid(conc_g, temp_g)
zz     = beta_hat[0] + beta_hat[1]*cc + beta_hat[2]*tt
ax.plot_surface(cc, tt, zz, alpha=0.25, color="firebrick")

ax.set_xlabel("Concentracion (g/L)")
ax.set_ylabel("Temperatura (grados C)")
ax.set_zlabel("Rendimiento (%)")
ax.set_title("Datos y plano de regresion ajustado (MLR)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
X_sm  = sm.add_constant(df[["concentracion", "temperatura"]])
modelo = sm.OLS(rendimiento, X_sm).fit()
print(modelo.summary())
print()

print("Valores clave extraidos del modelo:")
print(f"  beta_0_hat = {modelo.params['const']:.4f}")
print(f"  beta_1_hat = {modelo.params['concentracion']:.4f}  (concentracion)")
print(f"  beta_2_hat = {modelo.params['temperatura']:.4f}  (temperatura)")
print(f"  R^2        = {modelo.rsquared:.4f}")
print(f"  R^2_adj    = {modelo.rsquared_adj:.4f}")
print(f"  F_0        = {modelo.fvalue:.4f},  p = {modelo.f_pvalue:.4e}")
print(f"  RSE        = {np.sqrt(modelo.mse_resid):.4f}")
print()
print("Comparacion manual vs. statsmodels:")
print(f"  beta_0: manual = {beta_hat[0]:.4f},  sm = {modelo.params['const']:.4f}")
print(f"  beta_1: manual = {beta_hat[1]:.4f},  sm = {modelo.params['concentracion']:.4f}")
print(f"  beta_2: manual = {beta_hat[2]:.4f},  sm = {modelo.params['temperatura']:.4f}")

---
## 2. Bondad de Ajuste: $R^2$ y $R^2_{adj}$

La descomposicion $SS_T = SS_R + SS_E$ generaliza directamente al caso multiple.
Pero el $R^2$ tiene un problema critico en MLR:

> **Agregar cualquier predictor, aunque sea ruido puro, *siempre* aumenta $R^2$.**

Para comparar modelos con distinto numero de predictores se usa el
$R^2$ **ajustado** (M\&R, ec. 12.38):

$$R^2_{adj} = 1 - \frac{SS_E / (n - p - 1)}{SS_T / (n - 1)}
            = 1 - \frac{MS_E}{MS_T}$$

donde $p$ es el numero de predictores (sin el intercepto).

| Metrica | Formula | Cuando aumenta |
|---------|---------|----------------|
| $R^2$ | $1 - SS_E/SS_T$ | Siempre que se agrega un predictor |
| $R^2_{adj}$ | $1 - MS_E/MS_T$ | Solo si el predictor reduce $MS_E$ mas de lo que penaliza $df$ |
| RSE | $\sqrt{SS_E/(n-p-1)}$ | Estimador de $\sigma$ |

**Regla practica:** si $R^2_{adj}$ sube al agregar un predictor, el predictor
es util; si baja (aunque $R^2$ haya subido), el predictor introduce ruido.

In [ ]:
y_hat    = np.asarray(modelo.fittedvalues)
residuos = np.asarray(modelo.resid)
y_bar    = rendimiento.mean()

SS_T = np.sum((rendimiento - y_bar) ** 2)
SS_E = np.sum(residuos ** 2)
SS_R = SS_T - SS_E
p    = 2          # numero de predictores (sin intercepto)
MS_E = SS_E / (n - p - 1)
R2     = 1 - SS_E / SS_T
R2_adj = 1 - MS_E / (SS_T / (n - 1))
RSE    = np.sqrt(MS_E)

print("Descomposicion de la varianza (MLR con p=2):")
print(f"  SS_T  = {SS_T:.4f}   (df = {n-1})")
print(f"  SS_R  = {SS_R:.4f}   (df = {p})")
print(f"  SS_E  = {SS_E:.4f}   (df = {n-p-1})")
print()
print(f"  R^2      = {R2:.4f}   ({R2*100:.1f}% de la varianza explicada)")
print(f"  R^2_adj  = {R2_adj:.4f}")
print(f"  RSE      = {RSE:.4f}  (error tipico en %)")
print()
print("Verificacion statsmodels:")
print(f"  R^2 (modelo)     = {modelo.rsquared:.4f}")
print(f"  R^2_adj (modelo) = {modelo.rsquared_adj:.4f}")
print(f"  RSE (modelo)     = {np.sqrt(modelo.mse_resid):.4f}")

In [ ]:
# Demostrar que R^2 siempre sube pero R^2_adj puede bajar
np.random.seed(99)
ruido = np.random.normal(0, 1, n)

X_ruido  = sm.add_constant(pd.DataFrame({
    "concentracion": concentracion,
    "temperatura":   temperatura,
    "ruido":         ruido
}))
modelo_ruido = sm.OLS(rendimiento, X_ruido).fit()

print("Efecto de agregar un predictor de ruido puro:")
print()
print(f"  Modelo con conc + temp:")
print(f"    R^2     = {modelo.rsquared:.4f}")
print(f"    R^2_adj = {modelo.rsquared_adj:.4f}")
print(f"    AIC     = {modelo.aic:.2f}")
print()
print(f"  Modelo con conc + temp + ruido (predictor irrelevante):")
print(f"    R^2     = {modelo_ruido.rsquared:.4f}  <- subio (siempre sube)")
print(f"    R^2_adj = {modelo_ruido.rsquared_adj:.4f}  <- bajo  (penalizo el predictor inutil)")
print(f"    AIC     = {modelo_ruido.aic:.2f}")
print()
print("Conclusion: R^2_adj y AIC son mejores criterios para comparar modelos con diferente p.")

---
## 3. Inferencia

### 3.1 F-test global (significancia del modelo)

$$H_0: \beta_1 = \beta_2 = \cdots = \beta_p = 0
  \quad \text{(ningun predictor es util)}$$

$$F_0 = \frac{MS_R}{MS_E} = \frac{SS_R/p}{SS_E/(n-p-1)} \sim F_{p,\, n-p-1}
  \quad \text{bajo } H_0$$

### 3.2 t-tests individuales (efectos parciales)

Para cada coeficiente $\beta_j$ (con los demas en el modelo):

$$H_0: \beta_j = 0 \qquad
  T_0 = \frac{\hat{\beta}_j}{se(\hat{\beta}_j)} \sim t_{n-p-1}$$

donde $se(\hat{\beta}_j) = \hat{\sigma}\,\sqrt{C_{jj}}$ y $C_{jj}$ es el
$(j,j)$-elemento de $(\mathbf{X}^\top\mathbf{X})^{-1}$.

> **Diferencia clave con SLR:** en MLR el F-test y el t-test de cada coeficiente
> responden preguntas *distintas*.
> El F-test prueba si el modelo completo es significativo.
> Cada t-test prueba si ese predictor agrega informacion *dada la presencia de los demas*.
> Es posible que el F-test sea significativo pero algunos t-tests no lo sean
> (predictor redundante).

In [ ]:
print("--- F-test global ---")
print(f"  F_0   = {modelo.fvalue:.4f}")
print(f"  p     = {modelo.f_pvalue:.4e}")
print(f"  df    = ({p}, {n-p-1})")
if modelo.f_pvalue < 0.05:
    print("  Conclusion: el modelo es significativo globalmente (p < 0.05).")
print()

print("--- t-tests individuales ---")
ci_sm = np.asarray(modelo.conf_int())
for i, name in enumerate(modelo.params.index):
    print(f"  {name}:")
    print(f"    coef = {modelo.params[i]:.4f},  t = {modelo.tvalues[i]:.4f},  p = {modelo.pvalues[i]:.4e}")
    print(f"    IC 95%: [{ci_sm[i,0]:.4f}, {ci_sm[i,1]:.4f}]")
print()
print("Interpretacion de los coeficientes parciales:")
print(f"  Por cada g/L adicional de catalizador (temp. fija): rendimiento aumenta {modelo.params['concentracion']:.2f} pp")
print(f"  Por cada grado C adicional (concentracion fija):    rendimiento aumenta {modelo.params['temperatura']:.2f} pp")

---
## 4. Multicolinealidad y Factor de Inflacion de la Varianza (VIF)

**Multicolinealidad** ocurre cuando dos o mas predictores estan fuertemente
correlacionados entre si. Esto no sesga los estimadores, pero *infla* los
errores estandar, haciendo que los t-tests pierdan potencia y los coeficientes
se vuelvan inestables.

El **Factor de Inflacion de la Varianza** cuantifica cuanto aumenta la varianza
de $\hat{\beta}_j$ debido a la correlacion con los demas predictores (M\&R, ec. 12.43):

$$VIF_j = \frac{1}{1 - R^2_j}$$

donde $R^2_j$ es el $R^2$ de la regresion de $x_j$ sobre los demas predictores.

| $VIF_j$ | Diagnostico |
|---------|-------------|
| $\approx 1$ | Sin multicolinealidad |
| $1$ -- $5$ | Leve, generalmente aceptable |
| $5$ -- $10$ | Moderada, vigilar |
| $> 10$ | Alta: problema serio |

In [ ]:
# VIF para el modelo con concentracion y temperatura
# Nota: el VIF se calcula para los predictores (columnas 1 en adelante); se omite el intercepto.
X_vif = sm.add_constant(df[["concentracion", "temperatura"]])
vif_df = pd.DataFrame({
    "Variable": X_vif.columns[1:],
    "VIF":      [variance_inflation_factor(X_vif.values, i)
                 for i in range(1, X_vif.shape[1])]
})
print("VIF para los predictores del modelo:")
print(vif_df.to_string(index=False))
print()
print("Interpretacion: ambos VIF son cercanos a 1 (predictores independientes).")
print("No hay multicolinealidad problematica.")
print()

# Ejemplo de alta multicolinealidad: predictor casi duplicado de concentracion
np.random.seed(77)
concentracion_mod = concentracion + np.random.normal(0, 0.5, n)
X_col = sm.add_constant(pd.DataFrame({
    "concentracion":     concentracion,
    "temperatura":       temperatura,
    "conc_mod":          concentracion_mod
}))
vif_col = pd.DataFrame({
    "Variable": X_col.columns[1:],
    "VIF":      [variance_inflation_factor(X_col.values, i)
                 for i in range(1, X_col.shape[1])]
})
print("Ejemplo de alta multicolinealidad (conc + conc_mod muy correlacionadas):")
print(vif_col.to_string(index=False))
print()
r_cols, _ = stats.pearsonr(concentracion, concentracion_mod)
print(f"Correlacion entre concentracion y conc_mod: R = {r_cols:.4f}")
print("-> VIF elevado indica que los errores estandar de esos coeficientes son inestables.")

---
## 5. Seleccion de Variables

El principio de parsimonia dice: usar el **modelo mas simple que explique bien los datos**.

Agregar predictores siempre aumenta $R^2$, pero puede:
- Inflar los errores estandar (especialmente si hay multicolinealidad).
- Dificultar la interpretacion.
- Producir sobreajuste (mala generalizacion a datos nuevos).

**Eliminacion hacia atras (backward elimination):**
1. Ajusta el modelo completo con todos los predictores.
2. Elimina el predictor con el p-value mas alto (mayor a $\alpha = 0.05$).
3. Re-ajusta y repite hasta que todos los coeficientes sean significativos.

Otros criterios: **AIC** ($2p - 2\ln L$) y **BIC** ($p\ln n - 2\ln L$);
modelos mas simples reciben penalizacion menor.

In [ ]:
# Dataset de calidad de vino (Clase 14 -- regresion IV)
wine = pd.DataFrame({
    "oakiness": [3.9,3.7,3.5,3.9,3.7,3.4,3.9,3.8,3.7,4.0,
                 3.8,3.7,3.8,3.9,3.8,3.5,3.8,3.8,3.5,3.8,
                 4.0,3.8,3.7,3.9,3.9,3.8,3.8,3.5,3.9,3.6,
                 3.8,3.7,3.8,3.8,3.9,3.7,3.9,3.7],
    "flavor":   [3.7,3.8,3.8,3.8,3.7,3.8,3.8,3.6,3.8,3.8,
                 3.6,3.7,3.7,3.8,3.8,3.9,3.6,3.7,3.8,3.8,
                 3.7,3.7,3.8,3.8,3.7,3.6,3.8,3.9,3.7,3.9,
                 3.8,3.8,3.7,3.7,3.7,3.7,3.8,3.8],
    "body":     [3.3,3.5,3.4,3.5,3.2,3.4,3.5,3.0,3.5,3.6,
                 3.1,3.3,3.5,3.6,3.4,3.7,3.3,3.5,3.5,3.6,
                 3.5,3.2,3.3,3.6,3.8,3.5,3.5,3.8,3.6,3.9,
                 3.5,3.5,3.4,3.2,3.4,3.4,3.6,3.4],
    "aroma":    [3.8,4.1,3.7,4.0,3.7,3.9,4.0,3.6,3.8,4.1,
                 3.7,3.8,3.7,4.0,3.8,4.0,3.5,3.8,4.0,3.9,
                 3.9,3.7,3.6,3.8,3.9,3.8,3.7,4.1,3.9,4.0,
                 3.7,3.8,3.9,3.6,3.8,3.8,3.9,3.8],
    "clarity":  [3.0,3.4,3.4,3.5,3.2,3.5,3.4,3.3,3.4,3.5,
                 3.2,3.4,3.5,3.5,3.4,3.5,3.2,3.5,3.4,3.5,
                 3.5,3.3,3.2,3.5,3.5,3.5,3.5,3.7,3.4,3.7,
                 3.4,3.5,3.4,3.4,3.5,3.4,3.5,3.5],
    "quality":  [7.0,7.5,7.5,7.5,6.5,7.0,7.5,7.0,7.5,8.0,
                 6.5,7.0,7.5,8.0,7.5,8.0,7.0,7.5,7.5,7.5,
                 7.5,7.0,7.0,7.5,7.5,7.5,7.5,8.0,7.5,8.0,
                 7.5,7.5,7.5,7.0,7.5,7.0,7.5,7.5]
})

# Modelo completo: quality ~ oakiness + flavor + body + aroma + clarity
X_full  = sm.add_constant(wine.drop(columns="quality"))
m_full  = sm.OLS(wine["quality"], X_full).fit()
pred_cols = [c for c in X_full.columns if c != "const"]

print("Modelo completo (p = 5 predictores):")
print(f"  R^2 = {m_full.rsquared:.4f},  R^2_adj = {m_full.rsquared_adj:.4f}  AIC = {m_full.aic:.2f}")
print()
pvals = m_full.pvalues[pred_cols]
print("  p-values de los predictores:")
for name in pred_cols:
    flag = "<-- ELIMINAR (mayor p)" if name == pvals.idxmax() else ""
    print(f"    {name:10s}  p = {pvals[name]:.3f}  {flag}")
print()

# Paso 1: eliminar aroma (mayor p-value = 0.407)
X_red   = sm.add_constant(wine[["oakiness", "flavor", "body", "clarity"]])
m_red   = sm.OLS(wine["quality"], X_red).fit()
pvals_r = m_red.pvalues[["oakiness", "flavor", "body", "clarity"]]

print("Paso 1: eliminar 'aroma' (p = 0.407, el mas alto):")
print(f"  R^2 = {m_red.rsquared:.4f},  R^2_adj = {m_red.rsquared_adj:.4f}  AIC = {m_red.aic:.2f}")
print()
print("  p-values de los predictores restantes:")
for name in pvals_r.index:
    print(f"    {name:10s}  p = {pvals_r[name]:.3f}")
print()
all_sig = (pvals_r < 0.05).all()
print(f"  Todos los predictores son significativos (p < 0.05): {all_sig} -> PARAR")
print()
print("Modelo final: quality ~ oakiness + flavor + body + clarity")
print(f"  Perdida de R^2:    {m_full.rsquared - m_red.rsquared:.4f}  (minima)")
print(f"  Ganancia R^2_adj:  {m_red.rsquared_adj - m_full.rsquared_adj:.4f}  (el modelo es mas parsimonioso)")

---
## 6. Diagnosticos del Modelo

Los mismos diagnosticos que en SLR aplican directamente al caso multiple.
Hay que revisar:

| Supuesto | Grafico | Test formal |
|----------|---------|-------------|
| Normalidad de residuos | Q-Q plot | Shapiro-Wilk |
| Homocedasticidad | Residuos vs $\hat{y}$ | Breusch-Pagan |
| Independencia | Residuos vs orden | Durbin-Watson |
| Puntos influyentes | Distancia de Cook | $D_i > 4/n$ |

El residuo $\hat{\varepsilon}_i = y_i - \hat{y}_i$ debe lucir como una muestra
de $\mathcal{N}(0, \sigma^2)$ sin patrones sistematicos.

In [ ]:
influence   = modelo.get_influence()
cook_d, _   = influence.cooks_distance
umbral_cook = 4 / n

fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=150)

# Panel 1: residuos vs. predichos
axes[0].scatter(y_hat, residuos, color="steelblue", s=40, zorder=3)
axes[0].axhline(0, color="firebrick", linestyle="--")
axes[0].set_xlabel("Valores predichos")
axes[0].set_ylabel("Residuos")
axes[0].set_title("Residuos vs. Predichos")
axes[0].grid(alpha=0.3)

# Panel 2: Q-Q plot
(osm, osr), (slope, intercept, _) = stats.probplot(residuos, dist="norm")
axes[1].scatter(osm, osr, color="steelblue", s=40)
axes[1].plot(osm, slope * osm + intercept, color="firebrick")
axes[1].set_xlabel("Cuantiles teoricos")
axes[1].set_ylabel("Cuantiles observados")
axes[1].set_title("Q-Q plot de residuos")
axes[1].grid(alpha=0.3)

# Panel 3: distancia de Cook
colors_cook = ["firebrick" if d > umbral_cook else "steelblue" for d in cook_d]
axes[2].bar(range(n), cook_d, color=colors_cook, alpha=0.7)
axes[2].axhline(umbral_cook, color="firebrick", linestyle="--",
                label=f"Umbral = 4/n = {umbral_cook:.3f}")
axes[2].set_xlabel("Indice de observacion")
axes[2].set_ylabel("Distancia de Cook")
axes[2].set_title("Distancia de Cook")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle("Diagnosticos del modelo MLR", fontsize=12)
plt.tight_layout()
plt.show()

influyentes = np.where(cook_d > umbral_cook)[0]
if len(influyentes):
    print(f"Observaciones influyentes (D_i > {umbral_cook:.3f}): {influyentes}")
else:
    print(f"Ninguna observacion supera el umbral D_i > {umbral_cook:.3f}.")

In [ ]:
sw_stat, sw_p = stats.shapiro(residuos)
_, bp_p, _, _ = het_breuschpagan(residuos, modelo.model.exog)

print("Tests formales de supuestos (modelo MLR):")
print()
print(f"  Normalidad   (Shapiro-Wilk):  W = {sw_stat:.4f},  p = {sw_p:.4f}")
print(f"  Homocedasticidad (Breusch-Pagan):      p = {bp_p:.4f}")
print()

norm_ok = sw_p > 0.05
homo_ok = bp_p > 0.05
print("Conclusion:")
print(f"  Normalidad:       {'OK  (p > 0.05)' if norm_ok else 'VIOLADA (p <= 0.05)'}")
print(f"  Homocedasticidad: {'OK  (p > 0.05)' if homo_ok else 'VIOLADA (p <= 0.05)'}")

> **Como interpretar los graficos de diagnostico en MLR:**
>
> - **Residuos vs. Predichos:** patrones en forma de abanico indican
>   heterocedasticidad; curvaturas sugieren que falta un termino no lineal.
> - **Q-Q plot:** igual que en SLR; desviaciones en las colas indican no normalidad.
> - **Distancia de Cook:** puntos sobre $4/n$ merecen inspeccion.
>   En MLR son especialmente importantes porque la influencia de un punto
>   puede enmascararse si dos predictores estan correlacionados.
>
> Si los supuestos se violan, considere: transformaciones de $y$ (log, raiz),
> transformaciones de algun $x_j$, o eliminar observaciones atipicas
> solo si tienen una justificacion sustantiva.

---
## 7. Prediccion

Las formulas de prediccion de SLR se generalizan directamente.
Para un nuevo vector de predictores $\mathbf{x}_0 = (1, x_{01}, x_{02})$:

**IC para la respuesta media:**

$$\hat{y}_0 \pm t_{\alpha/2,\, n-p-1}\;\hat{\sigma}
  \sqrt{\mathbf{x}_0^\top (\mathbf{X}^\top\mathbf{X})^{-1} \mathbf{x}_0}$$

**IP para una nueva observacion:**

$$\hat{y}_0 \pm t_{\alpha/2,\, n-p-1}\;\hat{\sigma}
  \sqrt{1 + \mathbf{x}_0^\top (\mathbf{X}^\top\mathbf{X})^{-1} \mathbf{x}_0}$$

En `statsmodels` esto se calcula directamente con `model.get_prediction(new_X)`.

In [ ]:
# Predecir para x1 = 10 g/L, x2 = 70 grados C
x0_conc, x0_temp = 10.0, 70.0

new_obs = pd.DataFrame({"concentracion": [x0_conc], "temperatura": [x0_temp]})
new_X   = sm.add_constant(new_obs, has_constant="add")

pred         = modelo.get_prediction(new_X)
pred_summary = pred.summary_frame(alpha=0.05)

print(f"Prediccion para x1 = {x0_conc} g/L, x2 = {x0_temp} grados C:")
print()
print(f"  y_hat_0                     = {pred_summary['mean'].values[0]:.4f} %")
print(f"  IC 95% respuesta media:      [{pred_summary['mean_ci_lower'].values[0]:.4f}, {pred_summary['mean_ci_upper'].values[0]:.4f}] %")
print(f"  IP 95% observacion nueva:    [{pred_summary['obs_ci_lower'].values[0]:.4f},  {pred_summary['obs_ci_upper'].values[0]:.4f}] %")
print()
print(f"  Amplitud IC: {pred_summary['mean_ci_upper'].values[0] - pred_summary['mean_ci_lower'].values[0]:.4f} pp")
print(f"  Amplitud IP: {pred_summary['obs_ci_upper'].values[0] - pred_summary['obs_ci_lower'].values[0]:.4f} pp")
print()
print("El IP es siempre mas ancho que el IC (mismo principio que en SLR).")

---
## 8. Sintesis

### Flujo de trabajo recomendado para MLR

1. **EDA:** scatter de cada $x_j$ vs $y$; matriz de correlaciones entre predictores.
2. **Ajuste:** `sm.OLS(y, sm.add_constant(df[predictor_list])).fit()`.
3. **F-test global:** `model.fvalue`, `model.f_pvalue` -- es el modelo util en absoluto?
4. **t-tests individuales:** `model.tvalues`, `model.pvalues` -- cuales predictores son significativos?
5. **Bondad de ajuste:** $R^2_{adj}$ y RSE (no usar $R^2$ solo para comparar modelos).
6. **VIF:** detectar multicolinealidad; eliminar o combinar predictores con $VIF > 10$.
7. **Seleccion de variables:** eliminacion hacia atras por p-value o criterio AIC/BIC.
8. **Diagnosticos:** Q-Q (normalidad), residuos vs $\hat{y}$ (homocedasticidad), Cook (influencia).
9. **Prediccion:** `model.get_prediction(new_X).summary_frame()` entrega IC y IP.

### Tabla de referencia rapida

| Concepto | Formula / Definicion | Python |
|---|---|---|
| OLS matricial | $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ | `np.linalg.solve(X.T@X, X.T@y)` |
| Ajuste | `sm.OLS(y, sm.add_constant(X)).fit()` | |
| $R^2$ | $1 - SS_E/SS_T$ | `model.rsquared` |
| $R^2_{adj}$ | $1 - MS_E/MS_T$ | `model.rsquared_adj` |
| RSE | $\sqrt{SS_E/(n-p-1)}$ | `np.sqrt(model.mse_resid)` |
| F-test | $F_0 = MS_R/MS_E \sim F_{p, n-p-1}$ | `model.fvalue`, `model.f_pvalue` |
| t-test por coef. | $T_0 = \hat{\beta}_j/se(\hat{\beta}_j) \sim t_{n-p-1}$ | `model.tvalues`, `model.pvalues` |
| IC coef. | $\hat{\beta}_j \pm t_{\alpha/2}\,se(\hat{\beta}_j)$ | `model.conf_int()` |
| VIF | $1/(1-R^2_j)$; umbral 10 | `variance_inflation_factor(X, j)` |
| Prediccion | IC y IP en $\mathbf{x}_0$ | `model.get_prediction(new_X).summary_frame()` |
| Normalidad | Shapiro-Wilk | `stats.shapiro(resid)` |
| Homocedasticidad | Breusch-Pagan | `het_breuschpagan(resid, exog)` |
| Influencia | Cook $D_i > 4/n$ | `model.get_influence().cooks_distance` |

**Referencia:** Montgomery & Runger (2019), *Applied Statistics and Probability for Engineers*,
7a ed., Wiley. Capitulo 12.